In [14]:
import os
import time
import pandas as pd
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 1. SSL/인증서 에러 방지 (환경 변수 설정)
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''

def crawl_dunkin():
    chrome_options = Options()
    # chrome_options.add_argument("--headless") # 실제 리스트가 뜨는지 확인하려면 주석 유지
    chrome_options.add_argument('--ignore-certificate-errors')
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 15) # 대기 시간 15초

    url = "https://www.dunkindonuts.co.kr/store"
    driver.get(url)
    
    all_data = []
    regions = ["서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종", 
               "경기", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]

    try:
        for region in regions:
            # 2. 지역 선택 드롭다운 클릭
            arrow_xpath = "//*[local-name()='path' and @d='M2 1.5L7 6.5L12 1.5']/ancestor::button"
            dropdown = wait.until(EC.element_to_be_clickable((By.XPATH, arrow_xpath)))
            driver.execute_script("arguments[0].click();", dropdown)
            time.sleep(1)

            # 3. 드롭다운 내 지역(div[data-value]) 클릭
            region_xpath = f"//div[@role='option' and @data-value='{region}']"
            region_option = wait.until(EC.visibility_of_element_located((By.XPATH, region_xpath)))
            driver.execute_script("arguments[0].click();", region_option)
            time.sleep(0.5)

            # 4. '검색' 버튼 클릭 (data-gtm-click="스토어-검색")
            search_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[@data-gtm-click='스토어-검색']")))
            driver.execute_script("arguments[0].click();", search_btn)
            
            # [핵심] 5. 리스트(li)가 화면에 나타날 때까지 기다림
            # 보내주신 li 클래스명 중 'gtm-click'을 기준으로 기다립니다.
            try:
                wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "gtm-click")))
                time.sleep(2) # 리스트가 전체 다 렌더링될 때까지 여유 시간
            except:
                print(f"[{region}] 검색 결과가 없거나 로딩 지연 중")
                continue

            # 6. Selenium으로 직접 모든 <li> 태그 수집
            # ul.w-full 안에 있는 모든 li를 가져옵니다.
            li_elements = driver.find_elements(By.CSS_SELECTOR, "ul.w-full li")
            
            region_count = 0
            for li in li_elements:
                try:
                    # 매장명: li 안에서 'font-bold' 클래스를 가진 div 찾기
                    name = li.find_element(By.CLASS_NAME, "font-bold").text.strip()
                    # 주소: li 안에서 'leading-tight' 클래스를 가진 div 찾기
                    address = li.find_element(By.CLASS_NAME, "leading-tight").text.strip()
                    
                    if name:
                        all_data.append({
                            '매장명': name,
                            '주소': address,
                            '지역': region
                        })
                        region_count += 1
                except:
                    # 정보가 없는 li는 패스
                    continue

            # 8. 한 지역 완료 시 시간 및 수집 개수 콘솔 출력
            now = datetime.now().strftime('%H:%M:%S')
            print(f"[{now}] {region} 모든 <li> 수집 완료: {region_count}개")

        # 9. 최종 저장 (dunkindonuts.csv)
        if all_data:
            df = pd.DataFrame(all_data)
            # 중복 제거 (리스트 갱신 시 겹치는 데이터 방지)
            df = df.drop_duplicates(subset=['매장명', '주소'])
            df.to_csv("dunkindonuts.csv", index=False, encoding="utf-8-sig")
            print("-" * 50)
            print(f"🎉 성공! 총 {len(df)}개 매장 정보를 저장했습니다.")
        else:
            print("\n❌ 모든 시도에도 데이터가 0개입니다. 창이 떴을 때 리스트가 보였는지 확인해주세요.")

    except Exception as e:
        print(f"⚠️ 에러 발생: {e}")
    finally:
        driver.quit()

if __name__ == "__main__":
    crawl_dunkin()

[13:02:25] 서울 모든 <li> 수집 완료: 95개
[13:02:31] 부산 모든 <li> 수집 완료: 26개
[13:02:36] 대구 모든 <li> 수집 완료: 30개
[13:02:42] 인천 모든 <li> 수집 완료: 36개
[13:02:48] 광주 모든 <li> 수집 완료: 18개
[13:02:53] 대전 모든 <li> 수집 완료: 20개
[13:02:58] 울산 모든 <li> 수집 완료: 11개
[13:03:02] 세종 모든 <li> 수집 완료: 6개
[13:03:17] 경기 모든 <li> 수집 완료: 192개
[13:03:23] 강원 모든 <li> 수집 완료: 23개
[13:03:28] 충북 모든 <li> 수집 완료: 25개
[13:03:34] 충남 모든 <li> 수집 완료: 33개
[13:03:39] 전북 모든 <li> 수집 완료: 23개
[13:03:45] 전남 모든 <li> 수집 완료: 18개
[13:03:51] 경북 모든 <li> 수집 완료: 30개
[13:03:56] 경남 모든 <li> 수집 완료: 24개
[13:04:01] 제주 모든 <li> 수집 완료: 15개
--------------------------------------------------
🎉 성공! 총 625개 매장 정보를 저장했습니다.


In [15]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. CSV 파일 로드
file_path = 'dunkindonuts.csv'
df = pd.read_csv(file_path)

# 2. MySQL 연결 설정 (사용자 정보에 맞게 수정 필요)
user = 'root'      # MySQL 사용자명 (예: root)
password = '1234'  # MySQL 비밀번호
host = 'localhost'          # 호스트 (예: 127.0.0.1)
port = '3306'               # 포트 번호
db_name = 'coffee_store'    # 대상 데이터베이스

# 3. 데이터베이스 생성 (이미 존재하면 무시)
# 먼저 MySQL 서버 자체에 연결하여 DB가 없으면 생성합니다.
temp_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")
with temp_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {db_name}"))
    print(f"데이터베이스 '{db_name}'가 준비되었습니다.")

# 4. SQLAlchemy 엔진 생성 (coffee_store DB 연결)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}?charset=utf8mb4")

# 5. 데이터프레임을 MySQL 테이블로 저장
# - name: 생성할 테이블 이름 ('the_venti')
# - if_exists: 'fail' (이미 있으면 오류), 'replace' (기존 삭제 후 생성), 'append' (기존 테이블에 추가)
# - index: False (데이터프레임의 인덱스는 컬럼으로 넣지 않음)
try:
    df.to_sql(name='dunkindonuts', con=engine, if_exists='replace', index=False)
    print(f"테이블 'dunkindonuts'에 총 {len(df)}건의 데이터가 성공적으로 저장되었습니다.")
except Exception as e:
    print(f"데이터 저장 중 오류 발생: {e}")

# 연결 종료
engine.dispose()

데이터베이스 'coffee_store'가 준비되었습니다.
테이블 'dunkindonuts'에 총 625건의 데이터가 성공적으로 저장되었습니다.
